# Disclaimer: 
This version of eda.ipynb is here as a demonstration of how this work. To get the full functional notebook, please visit the following link: https://www.kaggle.com/code/thoangha/exploratory-data-analysis

In [ ]:
%pip install pandas matplotlib seaborn
%pip install --extra-index-url=https://pypi.nvidia.com cudf-cu12

In [ ]:
import pandas as pd
import cudf
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
# file_path = '/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json'
file_path = "path_to_your_file"

In [ ]:
def load_data(file_path, sample_size=1000000):
    """
        Reads a chunk of JSON lines and pushes it to the GPU.
    """

    data = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= sample_size:
                break
            data.append(json.loads(line))
    
    pdf = pd.DataFrame(data)
    return cudf.from_pandas(pdf)

In [ ]:
print("Loading data")
gdf = load_data(file_path)

In [ ]:
print("Extracting features and applying the 2015-2026 filter...")

gdf['year'] = gdf['update_date'].str.slice(0, 4).astype('int32')
gdf = gdf[(gdf['year'] >= 2015) & (gdf['year'] <= 2026)]

In [ ]:
# Extract primary category and domain for the filtered dataset
gdf['primary_category'] = gdf['categories'].str.split(' ').str[0]
gdf['domain'] = gdf['primary_category'].str.split('-').str[0].str.split('.').str[0]

In [ ]:
# Calculate abstract word count
gdf['abstract_word_count'] = gdf['abstract'].str.count(' ') + 1

# Extract all unique broad domains
unique_domains = gdf['domain'].unique().to_pandas().tolist()

In [ ]:
sns.set_theme(style="whitegrid")

# Visualise the Top 10 Research Domains (2015-2026)
plt.figure(figsize=(10, 6))
top_domains = gdf['domain'].value_counts().head(10).to_pandas()
sns.barplot(x=top_domains.values, y=top_domains.index, palette='mako')
plt.title('Top 10 Research Domains on arXiv (2015 - 2026)')
plt.xlabel('Number of Papers')
plt.ylabel('Domain Classification')
plt.tight_layout()
plt.show()

In [ ]:
# Visualise Paper Submissions Over Time
plt.figure(figsize=(10, 5))
yearly_counts = gdf.groupby('year').size().reset_index()
yearly_counts.columns = ['year', 'counts']
yearly_counts_cpu = yearly_counts.to_pandas().sort_values('year')
sns.barplot(data=yearly_counts_cpu, x='year', y='counts', palette='viridis')
plt.title('arXiv Papers (2015 - 2026)')
plt.xlabel('Year')
plt.ylabel('Total Papers')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of Abstract Word Counts
plt.figure(figsize=(10, 5))
# Transfer abstract word counts to Pandas for Seaborn plotting
word_counts_cpu = gdf['abstract_word_count'].to_pandas()
sns.histplot(word_counts_cpu, bins=60, kde=True, color='indigo')
plt.title('Distribution of Abstract Word Counts (2015 - 2026)')
plt.xlabel('Word Count')
plt.xlim(0, 400) 
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Define the complete taxonomy data
taxonomy_data = [
    {'Broad Domain Code': 'cond-mat', 'Broad Domain Name': 'Condensed Matter Physics', 'Sub-category Code': 'cond-mat.mes-hall', 'Full Sub-category Name': 'Mesoscale and Nanoscale Physics'},
    {'Broad Domain Code': 'cond-mat', 'Broad Domain Name': 'Condensed Matter Physics', 'Sub-category Code': 'cond-mat.mtrl-sci', 'Full Sub-category Name': 'Materials Science'},
    {'Broad Domain Code': 'cond-mat', 'Broad Domain Name': 'Condensed Matter Physics', 'Sub-category Code': 'cond-mat.str-el', 'Full Sub-category Name': 'Strongly Correlated Electrons'},
    {'Broad Domain Code': 'cond-mat', 'Broad Domain Name': 'Condensed Matter Physics', 'Sub-category Code': 'cond-mat.stat-mech', 'Full Sub-category Name': 'Statistical Mechanics'},
    {'Broad Domain Code': 'cond-mat', 'Broad Domain Name': 'Condensed Matter Physics', 'Sub-category Code': 'cond-mat.supr-con', 'Full Sub-category Name': 'Superconductivity'},
    {'Broad Domain Code': 'cond-mat', 'Broad Domain Name': 'Condensed Matter Physics', 'Sub-category Code': 'cond-mat.soft', 'Full Sub-category Name': 'Soft Condensed Matter'},
    {'Broad Domain Code': 'cond-mat', 'Broad Domain Name': 'Condensed Matter Physics', 'Sub-category Code': 'cond-mat.quant-gas', 'Full Sub-category Name': 'Quantum Gases'},
    {'Broad Domain Code': 'cond-mat', 'Broad Domain Name': 'Condensed Matter Physics', 'Sub-category Code': 'cond-mat.dis-nn', 'Full Sub-category Name': 'Disordered Systems and Neural Networks'},
    {'Broad Domain Code': 'cond-mat', 'Broad Domain Name': 'Condensed Matter Physics', 'Sub-category Code': 'cond-mat.other', 'Full Sub-category Name': 'Other Condensed Matter'},
    {'Broad Domain Code': 'hep', 'Broad Domain Name': 'High Energy Physics', 'Sub-category Code': 'hep-ph', 'Full Sub-category Name': 'High Energy Physics - Phenomenology'},
    {'Broad Domain Code': 'hep', 'Broad Domain Name': 'High Energy Physics', 'Sub-category Code': 'hep-th', 'Full Sub-category Name': 'High Energy Physics - Theory'},
    {'Broad Domain Code': 'hep', 'Broad Domain Name': 'High Energy Physics', 'Sub-category Code': 'hep-ex', 'Full Sub-category Name': 'High Energy Physics - Experiment'},
    {'Broad Domain Code': 'hep', 'Broad Domain Name': 'High Energy Physics', 'Sub-category Code': 'hep-lat', 'Full Sub-category Name': 'High Energy Physics - Lattice'},
    {'Broad Domain Code': 'astro-ph', 'Broad Domain Name': 'Astrophysics', 'Sub-category Code': 'astro-ph.SR', 'Full Sub-category Name': 'Solar and Stellar Astrophysics'},
    {'Broad Domain Code': 'astro-ph', 'Broad Domain Name': 'Astrophysics', 'Sub-category Code': 'astro-ph.GA', 'Full Sub-category Name': 'Astrophysics of Galaxies'},
    {'Broad Domain Code': 'astro-ph', 'Broad Domain Name': 'Astrophysics', 'Sub-category Code': 'astro-ph.CO', 'Full Sub-category Name': 'Cosmology and Nongalactic Astrophysics'},
    {'Broad Domain Code': 'astro-ph', 'Broad Domain Name': 'Astrophysics', 'Sub-category Code': 'astro-ph.HE', 'Full Sub-category Name': 'High Energy Astrophysical Phenomena'},
    {'Broad Domain Code': 'astro-ph', 'Broad Domain Name': 'Astrophysics', 'Sub-category Code': 'astro-ph.EP', 'Full Sub-category Name': 'Earth and Planetary Astrophysics'},
    {'Broad Domain Code': 'astro-ph', 'Broad Domain Name': 'Astrophysics', 'Sub-category Code': 'astro-ph.IM', 'Full Sub-category Name': 'Instrumentation and Methods for Astrophysics'},
    {'Broad Domain Code': 'quant-ph', 'Broad Domain Name': 'Quantum Physics', 'Sub-category Code': 'quant-ph', 'Full Sub-category Name': 'Quantum Physics'},
    {'Broad Domain Code': 'gr-qc', 'Broad Domain Name': 'General Relativity', 'Sub-category Code': 'gr-qc', 'Full Sub-category Name': 'General Relativity and Quantum Cosmology'},
    {'Broad Domain Code': 'nucl', 'Broad Domain Name': 'Nuclear Physics', 'Sub-category Code': 'nucl-th', 'Full Sub-category Name': 'Nuclear Theory'},
    {'Broad Domain Code': 'nucl', 'Broad Domain Name': 'Nuclear Physics', 'Sub-category Code': 'nucl-ex', 'Full Sub-category Name': 'Nuclear Experiment'},
    {'Broad Domain Code': 'nlin', 'Broad Domain Name': 'Nonlinear Sciences', 'Sub-category Code': 'nlin.CD', 'Full Sub-category Name': 'Chaotic Dynamics'},
    {'Broad Domain Code': 'nlin', 'Broad Domain Name': 'Nonlinear Sciences', 'Sub-category Code': 'nlin.SI', 'Full Sub-category Name': 'Exactly Solvable and Integrable Systems'},
    {'Broad Domain Code': 'nlin', 'Broad Domain Name': 'Nonlinear Sciences', 'Sub-category Code': 'nlin.PS', 'Full Sub-category Name': 'Pattern Formation and Solitons'},
    {'Broad Domain Code': 'nlin', 'Broad Domain Name': 'Nonlinear Sciences', 'Sub-category Code': 'nlin.AO', 'Full Sub-category Name': 'Adaptation and Self-Organising Systems'},
    {'Broad Domain Code': 'nlin', 'Broad Domain Name': 'Nonlinear Sciences', 'Sub-category Code': 'nlin.CG', 'Full Sub-category Name': 'Cellular Automata and Lattice Gases'},
    {'Broad Domain Code': 'stat', 'Broad Domain Name': 'Statistics', 'Sub-category Code': 'stat.ML', 'Full Sub-category Name': 'Machine Learning'},
    {'Broad Domain Code': 'stat', 'Broad Domain Name': 'Statistics', 'Sub-category Code': 'stat.ME', 'Full Sub-category Name': 'Methodology'},
    {'Broad Domain Code': 'stat', 'Broad Domain Name': 'Statistics', 'Sub-category Code': 'stat.AP', 'Full Sub-category Name': 'Applications'},
    {'Broad Domain Code': 'stat', 'Broad Domain Name': 'Statistics', 'Sub-category Code': 'stat.CO', 'Full Sub-category Name': 'Computation'},
    {'Broad Domain Code': 'stat', 'Broad Domain Name': 'Statistics', 'Sub-category Code': 'stat.OT', 'Full Sub-category Name': 'Other Statistics'},
    {'Broad Domain Code': 'eess', 'Broad Domain Name': 'Electrical Engineering', 'Sub-category Code': 'eess.SP', 'Full Sub-category Name': 'Signal Processing'},
    {'Broad Domain Code': 'eess', 'Broad Domain Name': 'Electrical Engineering', 'Sub-category Code': 'eess.IV', 'Full Sub-category Name': 'Image and Video Processing'},
    {'Broad Domain Code': 'eess', 'Broad Domain Name': 'Electrical Engineering', 'Sub-category Code': 'eess.AS', 'Full Sub-category Name': 'Audio and Speech Processing'},
    {'Broad Domain Code': 'eess', 'Broad Domain Name': 'Electrical Engineering', 'Sub-category Code': 'eess.SY', 'Full Sub-category Name': 'Systems and Control'},
    {'Broad Domain Code': 'econ', 'Broad Domain Name': 'Economics', 'Sub-category Code': 'econ.EM', 'Full Sub-category Name': 'Econometrics'},
    {'Broad Domain Code': 'econ', 'Broad Domain Name': 'Economics', 'Sub-category Code': 'econ.GN', 'Full Sub-category Name': 'General Economics'},
    {'Broad Domain Code': 'econ', 'Broad Domain Name': 'Economics', 'Sub-category Code': 'econ.TH', 'Full Sub-category Name': 'Theoretical Economics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.optics', 'Full Sub-category Name': 'Optics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.ins-det', 'Full Sub-category Name': 'Instrumentation and Detectors'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.flu-dyn', 'Full Sub-category Name': 'Fluid Dynamics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.atom-ph', 'Full Sub-category Name': 'Atomic Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.plasm-ph', 'Full Sub-category Name': 'Plasma Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.soc-ph', 'Full Sub-category Name': 'Physics and Society'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.chem-ph', 'Full Sub-category Name': 'Chemical Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.gen-ph', 'Full Sub-category Name': 'General Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.comp-ph', 'Full Sub-category Name': 'Computational Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.bio-ph', 'Full Sub-category Name': 'Biological Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.acc-ph', 'Full Sub-category Name': 'Accelerator Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.app-ph', 'Full Sub-category Name': 'Applied Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.class-ph', 'Full Sub-category Name': 'Classical Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.data-an', 'Full Sub-category Name': 'Data Analysis, Statistics and Probability'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.med-ph', 'Full Sub-category Name': 'Medical Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.hist-ph', 'Full Sub-category Name': 'History and Philosophy of Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.geo-ph', 'Full Sub-category Name': 'Geophysics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.ed-ph', 'Full Sub-category Name': 'Physics Education'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.ao-ph', 'Full Sub-category Name': 'Atmospheric and Oceanic Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.space-ph', 'Full Sub-category Name': 'Space Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.pop-ph', 'Full Sub-category Name': 'Popular Physics'},
    {'Broad Domain Code': 'physics', 'Broad Domain Name': 'General Physics', 'Sub-category Code': 'physics.atm-clus', 'Full Sub-category Name': 'Atomic and Molecular Clusters'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.AP', 'Full Sub-category Name': 'Analysis of PDEs'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.CO', 'Full Sub-category Name': 'Combinatorics'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.PR', 'Full Sub-category Name': 'Probability'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.AG', 'Full Sub-category Name': 'Algebraic Geometry'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math-ph', 'Full Sub-category Name': 'Mathematical Physics'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.NT', 'Full Sub-category Name': 'Number Theory'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.OC', 'Full Sub-category Name': 'Optimisation and Control'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.DG', 'Full Sub-category Name': 'Differential Geometry'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.NA', 'Full Sub-category Name': 'Numerical Analysis'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.DS', 'Full Sub-category Name': 'Dynamical Systems'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.FA', 'Full Sub-category Name': 'Functional Analysis'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.CA', 'Full Sub-category Name': 'Classical Analysis and ODEs'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.RT', 'Full Sub-category Name': 'Representation Theory'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.GT', 'Full Sub-category Name': 'Geometric Topology'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.ST', 'Full Sub-category Name': 'Statistics Theory'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.GR', 'Full Sub-category Name': 'Group Theory'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.RA', 'Full Sub-category Name': 'Rings and Algebras'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.CV', 'Full Sub-category Name': 'Complex Variables'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.LO', 'Full Sub-category Name': 'Logic'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.AT', 'Full Sub-category Name': 'Algebraic Topology'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.AC', 'Full Sub-category Name': 'Commutative Algebra'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.OA', 'Full Sub-category Name': 'Operator Algebras'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.MG', 'Full Sub-category Name': 'Metric Geometry'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.QA', 'Full Sub-category Name': 'Quantum Algebra'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.SP', 'Full Sub-category Name': 'Spectral Theory'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.SG', 'Full Sub-category Name': 'Symplectic Geometry'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.CT', 'Full Sub-category Name': 'Category Theory'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.GN', 'Full Sub-category Name': 'General Topology'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.GM', 'Full Sub-category Name': 'General Mathematics'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.HO', 'Full Sub-category Name': 'History and Overview'},
    {'Broad Domain Code': 'math', 'Broad Domain Name': 'Mathematics', 'Sub-category Code': 'math.KT', 'Full Sub-category Name': 'K-Theory and Homology'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.CV', 'Full Sub-category Name': 'Computer Vision and Pattern Recognition'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.IT', 'Full Sub-category Name': 'Information Theory'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.LG', 'Full Sub-category Name': 'Machine Learning'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.CL', 'Full Sub-category Name': 'Computation and Language'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.DS', 'Full Sub-category Name': 'Data Structures and Algorithms'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.NI', 'Full Sub-category Name': 'Networking and Internet Architecture'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.AI', 'Full Sub-category Name': 'Artificial Intelligence'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.CR', 'Full Sub-category Name': 'Cryptography and Security'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.SY', 'Full Sub-category Name': 'Systems and Control'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.DC', 'Full Sub-category Name': 'Distributed, Parallel, and Cluster Computing'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.LO', 'Full Sub-category Name': 'Logic in Computer Science'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.SI', 'Full Sub-category Name': 'Social and Information Networks'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.RO', 'Full Sub-category Name': 'Robotics'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.CY', 'Full Sub-category Name': 'Computers and Society'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.SE', 'Full Sub-category Name': 'Software Engineering'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.GT', 'Full Sub-category Name': 'Computer Science and Game Theory'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.DM', 'Full Sub-category Name': 'Discrete Mathematics'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.NE', 'Full Sub-category Name': 'Neural and Evolutionary Computing'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.IR', 'Full Sub-category Name': 'Information Retrieval'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.CC', 'Full Sub-category Name': 'Computational Complexity'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.DB', 'Full Sub-category Name': 'Databases'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.CG', 'Full Sub-category Name': 'Computational Geometry'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.PL', 'Full Sub-category Name': 'Programming Languages'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.HC', 'Full Sub-category Name': 'Human-Computer Interaction'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.DL', 'Full Sub-category Name': 'Digital Libraries'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.SD', 'Full Sub-category Name': 'Sound'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.FL', 'Full Sub-category Name': 'Formal Languages and Automata Theory'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.CE', 'Full Sub-category Name': 'Computational Engineering, Finance, and Science'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.ET', 'Full Sub-category Name': 'Emerging Technologies'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.NA', 'Full Sub-category Name': 'Numerical Analysis'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.MM', 'Full Sub-category Name': 'Multimedia'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.OH', 'Full Sub-category Name': 'Other Computer Science'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.GR', 'Full Sub-category Name': 'Graphics'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.MA', 'Full Sub-category Name': 'Multiagent Systems'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.AR', 'Full Sub-category Name': 'Hardware Architecture'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.SC', 'Full Sub-category Name': 'Symbolic Computation'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.MS', 'Full Sub-category Name': 'Mathematical Software'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.PF', 'Full Sub-category Name': 'Performance'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.OS', 'Full Sub-category Name': 'Operating Systems'},
    {'Broad Domain Code': 'cs', 'Broad Domain Name': 'Computer Science', 'Sub-category Code': 'cs.GL', 'Full Sub-category Name': 'General Literature'},
    {'Broad Domain Code': 'q-bio', 'Broad Domain Name': 'Quantitative Biology', 'Sub-category Code': 'q-bio.PE', 'Full Sub-category Name': 'Populations and Evolution'},
    {'Broad Domain Code': 'q-bio', 'Broad Domain Name': 'Quantitative Biology', 'Sub-category Code': 'q-bio.NC', 'Full Sub-category Name': 'Neurons and Cognition'},
    {'Broad Domain Code': 'q-bio', 'Broad Domain Name': 'Quantitative Biology', 'Sub-category Code': 'q-bio.QM', 'Full Sub-category Name': 'Quantitative Methods'},
    {'Broad Domain Code': 'q-bio', 'Broad Domain Name': 'Quantitative Biology', 'Sub-category Code': 'q-bio.MN', 'Full Sub-category Name': 'Molecular Networks'},
    {'Broad Domain Code': 'q-bio', 'Broad Domain Name': 'Quantitative Biology', 'Sub-category Code': 'q-bio.BM', 'Full Sub-category Name': 'Biomolecules'},
    {'Broad Domain Code': 'q-fin', 'Broad Domain Name': 'Quantitative Finance', 'Sub-category Code': 'q-fin.MF', 'Full Sub-category Name': 'Mathematical Finance'},
    {'Broad Domain Code': 'q-fin', 'Broad Domain Name': 'Quantitative Finance', 'Sub-category Code': 'q-fin.ST', 'Full Sub-category Name': 'Statistical Finance'},
    {'Broad Domain Code': 'q-bio', 'Broad Domain Name': 'Quantitative Biology', 'Sub-category Code': 'q-bio.GN', 'Full Sub-category Name': 'Genomics'},
    {'Broad Domain Code': 'q-fin', 'Broad Domain Name': 'Quantitative Finance', 'Sub-category Code': 'q-fin.GN', 'Full Sub-category Name': 'General Finance'},
    {'Broad Domain Code': 'q-fin', 'Broad Domain Name': 'Quantitative Finance', 'Sub-category Code': 'q-fin.EC', 'Full Sub-category Name': 'Economics'},
    {'Broad Domain Code': 'q-fin', 'Broad Domain Name': 'Quantitative Finance', 'Sub-category Code': 'q-fin.PR', 'Full Sub-category Name': 'Pricing of Securities'},
    {'Broad Domain Code': 'q-bio', 'Broad Domain Name': 'Quantitative Biology', 'Sub-category Code': 'q-bio.TO', 'Full Sub-category Name': 'Tissues and Organs'},
    {'Broad Domain Code': 'q-fin', 'Broad Domain Name': 'Quantitative Finance', 'Sub-category Code': 'q-fin.RM', 'Full Sub-category Name': 'Risk Management'},
    {'Broad Domain Code': 'q-bio', 'Broad Domain Name': 'Quantitative Biology', 'Sub-category Code': 'q-bio.CB', 'Full Sub-category Name': 'Cell Behaviour'},
    {'Broad Domain Code': 'q-fin', 'Broad Domain Name': 'Quantitative Finance', 'Sub-category Code': 'q-fin.PM', 'Full Sub-category Name': 'Portfolio Management'},
    {'Broad Domain Code': 'q-fin', 'Broad Domain Name': 'Quantitative Finance', 'Sub-category Code': 'q-fin.CP', 'Full Sub-category Name': 'Computational Finance'},
    {'Broad Domain Code': 'q-fin', 'Broad Domain Name': 'Quantitative Finance', 'Sub-category Code': 'q-fin.TR', 'Full Sub-category Name': 'Trading and Market Microstructure'},
    {'Broad Domain Code': 'q-bio', 'Broad Domain Name': 'Quantitative Biology', 'Sub-category Code': 'q-bio.SC', 'Full Sub-category Name': 'Subcellular Processes'},
    {'Broad Domain Code': 'q-bio', 'Broad Domain Name': 'Quantitative Biology', 'Sub-category Code': 'q-bio.OT', 'Full Sub-category Name': 'Other Quantitative Biology'}
]

# Convert to a Pandas DataFrame
taxonomy_df = pd.DataFrame(taxonomy_data)

# Export to CSV in the local directory
csv_filename = 'arxiv_taxonomy.csv'
taxonomy_df.to_csv(csv_filename, index=False)
print(f"Success! Taxonomy has been saved to {csv_filename}")

# Display the dataframe properly
display(taxonomy_df)

In [ ]:
# Iterate through each domain automatically
for domain in unique_domains:
    
    # Filter the dataset for the current domain in the loop
    domain_gdf = gdf[gdf['domain'] == domain]
    
    # Count the specific sub-categories
    sub_categories = domain_gdf['primary_category'].value_counts().to_pandas()
    
    # Plotting logic
    plt.figure(figsize=(10, 6))
    sns.barplot(x=sub_categories.values, y=sub_categories.index, palette='crest')
    
    # Dynamically inject the domain name into the title
    plt.title(f'Breakdown of Sub-categories within {domain.upper()} (2015 - 2026)')
    plt.xlabel('Number of Papers')
    plt.ylabel('Primary Category (Sub-domain)')
    plt.tight_layout()
    plt.show()

In [ ]:
print("\n--- Filtered Dataset Summary (2015-2026) ---")
print("Total papers in this timeframe:", len(gdf))

print("\n--- Missing Metadata Check ---")
cols_to_check = ['doi', 'journal-ref', 'comments']
print(gdf[cols_to_check].isnull().sum().to_pandas())

print("\n--- Abstract Length Statistics ---")
print(gdf['abstract_word_count'].describe().to_pandas())